# 03 — ALT Label Merge

**Prerequisites:** notebooks 01 and 02 complete.

**Purpose:**
- Parse all five literature tables and ALTitude table1 to extract ALT calls.
- Build a long-form per-sample ALT evidence table with consensus columns.
- Flag conflicting calls for manual review.
- Merge final `alt_status`, `alt_source`, `source_confidence` into the manifest.

**Manual review checkpoint:** reviewer must resolve all `conflict_flag == True` rows.

In [ ]:
import sys, random, time
from pathlib import Path

random.seed(42)
RUN_TIMESTAMP = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
assert (ROOT / 'src').is_dir(), f"Cannot locate src/ — tried {_cwd} and {_cwd.parent}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT: {ROOT}')

from src.utils import log_versions
import pandas as pd
import numpy as np

vers = log_versions()
print(f'Run: {RUN_TIMESTAMP}')

## Load all ALT evidence sources

In [7]:
from src.manifest_builder import (
    load_de_nonneville,
    load_lawlor_pdx_labels,
    load_taglialatela,
    load_altitude_cellines,
    load_chen_2014_pcgp,
)

# Load the manifest FIRST so we can filter the long-form ALT table to samples
# that actually appear in our cohort. Without this, ALTitude's 959-line panel
# (mostly non-osteosarcoma cell lines) drowns the consensus computation in
# alt_unknown rows that have nothing to do with our manifest.
MANIFEST_PATH = ROOT / 'data' / 'manifests' / 'master_manifest.csv'
manifest = pd.read_csv(MANIFEST_PATH, dtype=str)
manifest_sample_ids = set(manifest['sample_id'].tolist())

de_non    = load_de_nonneville()
lawlor    = load_lawlor_pdx_labels()
tagliala  = load_taglialatela()
altitude  = load_altitude_cellines()
pcgp      = load_chen_2014_pcgp()  # Chen 2014 telomere-FISH ALT calls; previously missing from notebook 03

ALT_COLS = ['sample_id', 'cohort', 'alt_status', 'alt_source', 'source_confidence', 'alt_assay_evidence']

sources = [
    ('de_Nonneville_2022', de_non),
    ('Lawlor_2026_GCCRI',  lawlor),
    ('Taglialatela_2026',  tagliala),
    ('ALTitude_DepMap',    altitude),
    ('PCGP_2014_Chen',     pcgp),
]

long_rows = []
for source_name, df in sources:
    for col in ALT_COLS:
        if col not in df.columns:
            df[col] = 'unknown'
    sub = df[ALT_COLS].copy()
    sub['evidence_source'] = source_name
    long_rows.append(sub)

alt_long_unfiltered = pd.concat(long_rows, ignore_index=True)
print(f'Long-form ALT evidence (all sources): {len(alt_long_unfiltered)} rows')
print(alt_long_unfiltered['alt_status'].value_counts().to_string())

# Restrict to samples in the manifest. This drops ALTitude rows for cell lines
# we never include in the cohort (mostly non-osteosarcoma) but keeps every
# ALTitude row whose sample_id matches a DepMap OS line.
alt_long = alt_long_unfiltered[
    alt_long_unfiltered['sample_id'].isin(manifest_sample_ids)
].copy()
n_dropped = len(alt_long_unfiltered) - len(alt_long)
print(f'\nFiltered to manifest samples: {len(alt_long)} rows '
      f'({n_dropped:,} non-manifest rows dropped — mostly non-OS ALTitude lines)')
print(alt_long['alt_status'].value_counts().to_string())
print(f'\nUnique sample_ids in filtered alt_long: {alt_long["sample_id"].nunique()} '
      f'(out of {len(manifest_sample_ids)} in manifest)')

Long-form ALT evidence (all sources): 1039 rows
alt_status
alt_unknown               980
alt_gold_positive          48
alt_negative_high_conf     11

Filtered to manifest samples: 111 rows (928 non-manifest rows dropped — mostly non-OS ALTitude lines)
alt_status
alt_unknown               52
alt_gold_positive         48
alt_negative_high_conf    11

Unique sample_ids in filtered alt_long: 110 (out of 114 in manifest)


/Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(m

## Compute per-sample consensus columns

In [8]:
GOLD_POSITIVE = {'alt_gold_positive'}
GOLD_NEGATIVE = {'alt_negative_high_conf'}
SILVER        = {'alt_silver'}

def _consensus(group):
    statuses    = set(group['alt_status'].dropna().unique()) - {'alt_unknown', 'unknown', 'nan'}
    confidences = set(group['source_confidence'].dropna().unique())

    n_gold_pos = int((group['alt_status'].isin(GOLD_POSITIVE)).sum())
    n_gold_neg = int((group['alt_status'].isin(GOLD_NEGATIVE)).sum())
    n_silver   = int((group['alt_status'].isin(SILVER)).sum())
    n_sources  = int(group['evidence_source'].nunique())

    pos_s    = statuses & (GOLD_POSITIVE | SILVER)
    neg_s    = statuses & GOLD_NEGATIVE
    conflict = bool(pos_s and neg_s)

    if 'peer_reviewed' in confidences:
        conf_max = 'peer_reviewed'
    elif 'preprint' in confidences:
        conf_max = 'preprint'
    else:
        conf_max = 'unknown'

    if n_gold_pos > 0 and not conflict:
        consensus = 'alt_gold_positive'
    elif n_gold_neg > 0 and not conflict:
        consensus = 'alt_negative_high_conf'
    elif n_silver > 0 and not conflict:
        consensus = 'alt_silver'
    else:
        consensus = 'alt_unknown'

    return pd.Series({
        'n_gold_assays_positive': n_gold_pos,
        'n_gold_assays_negative': n_gold_neg,
        'n_silver_calls':         n_silver,
        'n_sources':              n_sources,
        'conflict_flag':          conflict,
        'source_confidence_max':  conf_max,
        'consensus_alt_status':   consensus,
        'all_sources':            '|'.join(sorted(group['alt_source'].dropna().unique())),
    })


consensus_df = alt_long.groupby('sample_id').apply(_consensus).reset_index()
print(f'Consensus rows: {len(consensus_df)}')
print(consensus_df['consensus_alt_status'].value_counts().to_string())

Consensus rows: 110
consensus_alt_status
alt_unknown               52
alt_gold_positive         48
alt_negative_high_conf    10


## Review conflicts

**Manual review checkpoint:** resolve each row where `conflict_flag == True`.

In [9]:
conflicts = consensus_df[consensus_df['conflict_flag'] == True]
print(f'Conflicting samples: {len(conflicts)}')

if len(conflicts) > 0:
    for sid in conflicts['sample_id'].tolist():
        print(f'\n--- {sid} ---')
        sub = alt_long[alt_long['sample_id'] == sid]
        print(sub[['evidence_source','alt_status','source_confidence','alt_assay_evidence']].to_string(index=False))
else:
    print('No conflicts — proceed.')

Conflicting samples: 0
No conflicts — proceed.


In [10]:
# ── MANUAL REVIEW CHECKPOINT ──────────────────────────────────────────────
# For each conflicting sample_id above, enter a final call with rationale.

CONFLICT_RESOLUTIONS = {
    # 'SJSA1': {
    #     'final_alt_status': 'alt_negative_high_conf',
    #     'rationale': 'ALTitude negative (peer-reviewed) outweighs Taglialatela implied call',
    # },
}

print(f'Resolutions provided for {len(CONFLICT_RESOLUTIONS)} samples.')
if len(conflicts) > 0 and len(CONFLICT_RESOLUTIONS) < len(conflicts):
    print('WARNING: Not all conflicts resolved. Unresolved samples will retain alt_unknown.')

Resolutions provided for 0 samples.


## Merge ALT calls into manifest

In [11]:
# `manifest` was loaded at the top of the notebook; reload it here in case
# this cell is re-run after an interactive edit.
MANIFEST_PATH = ROOT / 'data' / 'manifests' / 'master_manifest.csv'
manifest = pd.read_csv(MANIFEST_PATH, dtype=str)

# ALT-status priority: only overwrite an existing manifest call when the new
# consensus call is at least as informative. This is critical because the
# upstream loaders (load_chen_2014_pcgp, load_lawlor_pdx_labels, etc.) already
# wrote informative ALT calls into the manifest in notebook 01; without a
# priority gate, an alt_unknown consensus from ALTitude would silently
# clobber a peer-reviewed alt_gold_positive from telomere FISH.
ALT_PRIORITY = {
    'alt_unknown':            0,
    'unknown':                0,
    'alt_silver':             1,
    'alt_proxy':              1,
    'alt_negative_high_conf': 2,
    'alt_gold_positive':      2,
}

merged_count, skipped_unknown, skipped_downgrade, conflict_appended = 0, 0, 0, 0

for _, row in consensus_df.iterrows():
    sid      = row['sample_id']
    call     = row['consensus_alt_status']
    conf     = row['source_confidence_max']
    srcs     = row['all_sources']
    conflict = row['conflict_flag']

    if sid in CONFLICT_RESOLUTIONS:
        call = CONFLICT_RESOLUTIONS[sid]['final_alt_status']
        conf = 'peer_reviewed'

    mask = manifest['sample_id'] == sid
    if mask.sum() == 0:
        continue  # sample not in manifest

    current_call    = manifest.loc[mask, 'alt_status'].iloc[0]
    current_priority = ALT_PRIORITY.get(current_call, 0)
    new_priority     = ALT_PRIORITY.get(call, 0)

    if new_priority > current_priority:
        # Strict upgrade — replace the full triple.
        manifest.loc[mask, 'alt_status']        = call
        manifest.loc[mask, 'alt_source']        = srcs
        manifest.loc[mask, 'source_confidence'] = conf
        merged_count += 1
    elif new_priority == current_priority and new_priority > 0:
        # Same informative level — refresh source list (may now include more
        # corroborating evidence) and upgrade source_confidence only if the
        # new confidence is strictly stronger.
        manifest.loc[mask, 'alt_source'] = srcs
        cur_conf = manifest.loc[mask, 'source_confidence'].iloc[0]
        conf_rank = {'unknown': 0, 'preprint': 1, 'peer_reviewed': 2}
        if conf_rank.get(conf, 0) > conf_rank.get(cur_conf, 0):
            manifest.loc[mask, 'source_confidence'] = conf
        merged_count += 1
    else:
        # Either a downgrade (informative -> alt_unknown) or both are
        # alt_unknown. Skip — never write a less-informative call, and never
        # stamp source_confidence='peer_reviewed' onto a sample whose ALT
        # status is genuinely unknown (the ALTitude noise case).
        if new_priority == 0:
            skipped_unknown += 1
        else:
            skipped_downgrade += 1

    if conflict and sid not in CONFLICT_RESOLUTIONS:
        manifest.loc[mask, 'notes'] = (
            manifest.loc[mask, 'notes'].astype(str) + ' | ALT_CONFLICT_UNRESOLVED'
        )
        conflict_appended += 1

print(f'Merged ALT calls into manifest: {merged_count}')
print(f'  skipped (consensus = alt_unknown, not informative): {skipped_unknown}')
print(f'  skipped (would downgrade an existing informative call): {skipped_downgrade}')
print(f'  unresolved-conflict notes appended: {conflict_appended}')

# Recompute inclusion flags now that alt_status may have changed.
from src.label_rules import compute_inclusion_flags
flag_cols = [c for c in manifest.columns if c.startswith('include_')]
flag_rows = manifest.apply(compute_inclusion_flags, axis=1)
for col in flag_cols:
    manifest[col] = [r[col] for r in flag_rows]

manifest.to_csv(MANIFEST_PATH, index=False)
print(f'\nManifest written: {MANIFEST_PATH}')
print('\n=== Post-merge alt_status distribution ===')
print(manifest['alt_status'].value_counts(dropna=False).to_string())
print('\n=== Post-merge alt_status by cohort ===')
print(pd.crosstab(manifest['cohort'], manifest['alt_status'], dropna=False).to_string())
print('\n=== Post-merge source_confidence by alt_status ===')
print(pd.crosstab(manifest['alt_status'], manifest['source_confidence'], dropna=False).to_string())

Merged ALT calls into manifest: 58
  skipped (consensus = alt_unknown, not informative): 52
  skipped (would downgrade an existing informative call): 0
  unresolved-conflict notes appended: 0

Manifest written: /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline/data/manifests/master_manifest.csv

=== Post-merge alt_status distribution ===
alt_status
alt_unknown               56
alt_gold_positive         48
alt_negative_high_conf    11

=== Post-merge alt_status by cohort ===
alt_status          alt_gold_positive  alt_negative_high_conf  alt_unknown
cohort                                                                    
DepMap_26Q1                         0                       0           35
Lawlor_2026_GCCRI                  16                       1            0
PCGP_2014                          11                       2           21
Taglialatela_2026                   5                       2            0
de_Nonneville_2022  

## Notebook summary

In [12]:
report_lines = [
    '# Notebook 03 — ALT Label Merge Report',
    '',
    f'Run: {RUN_TIMESTAMP}',
    '',
    '## ALT status distribution after merge',
    manifest['alt_status'].value_counts().to_string(),
    '',
    f'## Conflicts found: {len(conflicts)}',
    f'## Conflicts resolved manually: {len(CONFLICT_RESOLUTIONS)}',
    '',
    '## Source confidence',
    manifest['source_confidence'].value_counts().to_string(),
    '',
    '## Manual review checkpoint',
    'All conflict_flag rows must be reviewed before proceeding to notebook 04.',
]
report_path = ROOT / 'data' / 'processed' / 'notebook_03_report.md'
report_path.write_text('\n'.join(report_lines))
print('\n'.join(report_lines))

# Notebook 03 — ALT Label Merge Report

Run: 2026-05-01T21:28:28Z

## ALT status distribution after merge
alt_status
alt_unknown               56
alt_gold_positive         48
alt_negative_high_conf    11

## Conflicts found: 0
## Conflicts resolved manually: 0

## Source confidence
source_confidence
peer_reviewed    56
unknown          35
preprint         24

## Manual review checkpoint
All conflict_flag rows must be reviewed before proceeding to notebook 04.
